# Customer Subscription Renewal — Machine Learning Data Preparation

## Project overview
This educational project prepares **synthetic** customer and renewal-history records for a future binary classification task. It demonstrates inspection, missing-data handling, merging, categorical encoding, separating labeled and unlabeled records, feature/target selection, and validation.

**Scope:** Data preparation only. **No model is trained, tested, or evaluated.** Five labeled examples are not enough to estimate predictive performance.

**Tools:** Python, pandas, NumPy, Jupyter Notebook.

## 1. Load and inspect the data

Two small synthetic tables represent customer attributes and observed renewal history. Customer 106 has no history record. We inspect both tables before making changes.

In [1]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "CustomerID": [101, 102, 103, 104, 105, 106],
    "Age": [25, 32, np.nan, 41, 29, 36],
    "City": [
        "Ottawa", "Toronto", "Ottawa",
        "Montreal", np.nan, "Toronto"
    ],
    "Subscription": [
        "Premium", "Basic", "Premium",
        "Basic", "Premium", "Basic"
    ]
})

history = pd.DataFrame({
    "CustomerID": [101, 102, 103, 104, 105],
    "MonthsActive": [12, 6, 18, 3, 9],
    "Renewed": ["Yes", "No", "Yes", "No", "Yes"]
})

In [2]:
display(customers)
display(history)

,CustomerID,Age,City,Subscription
0,101,25.0,Ottawa,Premium
1,102,32.0,Toronto,Basic
2,103,NaN,Ottawa,Premium
3,104,41.0,Montreal,Basic
4,105,29.0,NaN,Premium
5,106,36.0,Toronto,Basic


,CustomerID,MonthsActive,Renewed
0,101,12,Yes
1,102,6,No
2,103,18,Yes
3,104,3,No
4,105,9,Yes


In [3]:
print("Missing customer values:", customers.isna().sum())
print("Missing history values:", history.isna().sum())

Missing customer values: CustomerID      0
Age             1
City            1
Subscription    0
dtype: int64
Missing history values: CustomerID      0
MonthsActive    0
Renewed         0
dtype: int64


## 2. Clean customer attributes

For this **data-preparation demonstration**, replace the missing age with the mean of available ages and missing city with the explicit category `Unknown`. Work on a copy to preserve the raw data.

**Important modeling limitation:** This mean is calculated using the full customer table, **not** a training-only partition. That is acceptable for illustrating pandas transformations, but **must not be reused as a leakage-free train/test modeling pipeline**. In a real project, split data first and fit an imputer on training data only; apply the fitted imputer to validation, test, and new records.

In [4]:
mean_age = customers["Age"].mean()
customers_clean = customers.copy()
customers_clean["Age"] = customers_clean["Age"].fillna(mean_age)
customers_clean["City"] = customers_clean["City"].fillna("Unknown")
print("Mean age used for this demonstration:", mean_age)
display(customers_clean)

Mean age used for this demonstration: 32.6


,CustomerID,Age,City,Subscription
0,101,25.0,Ottawa,Premium
1,102,32.0,Toronto,Basic
2,103,32.6,Ottawa,Premium
3,104,41.0,Montreal,Basic
4,105,29.0,Unknown,Premium
5,106,36.0,Toronto,Basic


## 3. Merge customer attributes with renewal history

A **left join** retains all six customers, including customer 106, who has no matching renewal history. `validate="one_to_one"` checks merge cardinality; `_merge` records the join outcome for auditing. A missing renewal outcome must **not** be silently interpreted as a non-renewal.

In [5]:
merged_customers = pd.merge(
    customers_clean,
    history,
    on = ["CustomerID"],
    how="left",
    validate="one_to_one",
    indicator=True
)
    
display(merged_customers)

,CustomerID,Age,City,Subscription,MonthsActive,Renewed,_merge
0,101,25.0,Ottawa,Premium,12.0,Yes,both
1,102,32.0,Toronto,Basic,6.0,No,both
2,103,32.6,Ottawa,Premium,18.0,Yes,both
3,104,41.0,Montreal,Basic,3.0,No,both
4,105,29.0,Unknown,Premium,9.0,Yes,both
5,106,36.0,Toronto,Basic,NaN,NaN,left_only


## 4. Encode categorical features and renewal labels

One-hot encode `Subscription` and `City`, including `City_Unknown`. Map `Renewed` from `Yes`/`No` to `WillRenew` = 1/0. The unmatched customer retains a missing target. `Renewed` is the original target representation and must not be used as an input feature.

In [6]:
model_data = pd.get_dummies(
    merged_customers,
    columns=["Subscription", "City"],
    dtype=int
).copy()
model_data["WillRenew"] = model_data["Renewed"].map({"Yes": 1, "No": 0})
display(model_data)

,CustomerID,Age,MonthsActive,Renewed,_merge,Subscription_Basic,Subscription_Premium,City_Montreal,City_Ottawa,City_Toronto,City_Unknown,WillRenew
0,101,25.0,12.0,Yes,both,0,1,0,1,0,0,1.0
1,102,32.0,6.0,No,both,1,0,0,0,1,0,0.0
2,103,32.6,18.0,Yes,both,0,1,0,1,0,0,1.0
3,104,41.0,3.0,No,both,1,0,1,0,0,0,0.0
4,105,29.0,9.0,Yes,both,0,1,0,0,0,1,1.0
5,106,36.0,NaN,NaN,left_only,1,0,0,0,1,0,NaN


## 5. Separate labeled and unlabeled customers

Only customers with known renewal outcomes can supply supervised-learning labels. Keep customer 106 separately for possible future prediction; do not fabricate its outcome.

In [7]:
labeled_customers = model_data[model_data["WillRenew"].notna()].copy()
unlabeled_customers = model_data[model_data["WillRenew"].isna()].copy()
print("Labeled customer IDs:", labeled_customers["CustomerID"].tolist())
print("Unlabeled customer IDs:", unlabeled_customers["CustomerID"].tolist())

Labeled customer IDs: [101, 102, 103, 104, 105]
Unlabeled customer IDs: [106]


## 6. Select features (X) and target (y)

Use `Age`, subscription indicators, and city indicators as features. Exclude `CustomerID` (identifier), `_merge` (merge audit), `Renewed` (direct target leakage), and `WillRenew` (the target itself). **Also exclude `MonthsActive` until its measurement timing is verified**: it would be leakage if it incorporates activity after the prediction date.

Both subscription indicators are retained for teaching purposes; for some linear models, one can be dropped to avoid redundant predictors.

In [8]:
feature_columns = [
    "Age", "Subscription_Basic", "Subscription_Premium",
    "City_Montreal", "City_Ottawa", "City_Toronto", "City_Unknown"
]
X = labeled_customers[feature_columns].copy()
y = labeled_customers["WillRenew"].astype(int).copy()
display(X)
display(y)

,Age,Subscription_Basic,Subscription_Premium,City_Montreal,City_Ottawa,City_Toronto,City_Unknown
0,25.0,0,1,0,1,0,0
1,32.0,1,0,0,0,1,0
2,32.6,0,1,0,1,0,0
3,41.0,1,0,1,0,0,0
4,29.0,0,1,0,0,0,1


0    1
1    0
2    1
3    0
4    1
Name: WillRenew, dtype: int64

## 7. Validate the prepared data

Check row counts, feature completeness, target completeness, allowed labels, and row-index alignment. Class counts and proportions describe this tiny synthetic sample, **not** population renewal rates or predictive performance.

In [9]:
assert X.shape == (5, 7)
assert y.shape == (5,)
assert X.index.equals(y.index), "Features and targets are misaligned"
assert not X.isna().any().any(), "Missing values in features"
assert not y.isna().any(), "Missing target values"
assert set(y.unique()).issubset({0, 1}), "Unexpected target labels"
assert len(unlabeled_customers) == 1
assert unlabeled_customers["CustomerID"].tolist() == [106]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Missing feature values:", int(X.isna().sum().sum()))
print("Missing target values:", int(y.isna().sum()))
print("Renewal counts:", y.value_counts().sort_index())
print("Renewal proportions:", y.value_counts(normalize=True).sort_index())
print("All validation checks passed.")

Feature matrix shape: (5, 7)
Target shape: (5,)
Missing feature values: 0
Missing target values: 0
Renewal counts: WillRenew
0    2
1    3
Name: count, dtype: int64
Renewal proportions: WillRenew
0    0.4
1    0.6
Name: proportion, dtype: float64
All validation checks passed.


## Results

- Five labeled customers were prepared with seven selected numeric features.
- Three labeled customers (60%) renewed; two (40%) did not.
- One customer (ID 106) has no known renewal outcome and remains unlabeled.
- The selected feature matrix and target contain no missing values, and labels are restricted to 0 and 1.

## Limitations

1. **Insufficient sample size:** Five labeled, synthetic observations cannot establish generalizable predictive performance. Adding fabricated observations would not resolve this limitation.
2. **Demonstration-only imputation:** Age was imputed using the available ages in the full synthetic table. For future predictive modeling, split into training/test sets first and fit preprocessing **only on training data** (e.g., `SimpleImputer` inside a scikit-learn pipeline).
3. **Feature timing:** `MonthsActive` is excluded pending verification that it would be available before the renewal decision.
4. **Potential redundancy:** Both subscription indicator columns are retained for clarity, though one may be redundant in certain models.
5. **No model evaluation:** No training, test split, accuracy, or performance claims are made.

## Conclusion

This notebook demonstrates a reproducible pandas workflow for inspecting, cleaning, merging, encoding, selecting, and validating customer data. The resulting `X` and `y` are **illustrative outputs**, not evidence of a production-ready renewal prediction model.

**Next step:** Obtain a larger, representative dataset with timestamps, establish a prediction cutoff, split training and test data, fit preprocessing on training data only, and then train and evaluate a classifier.